# 02 — Exploratory Data Analysis

Ports the six-question EDA template from Fares Sayah's Kaggle notebook to NFLX and its peers,
and adds the distributional analysis that template leaves out.

The questions:

1. How has the price moved, and how does it compare to peers?
2. What do the moving averages show?
3. What does the daily return distribution look like — and is it normal?
4. How correlated is NFLX with its peers?
5. How much risk are we taking for the return?

The answer to question 3 is the one that matters most for the rest of the project.

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src import config, plots
from src.data import load_prices, load_many, closing_prices
from src.features import (
    daily_returns, log_returns, add_moving_averages,
    rolling_volatility, return_summary, tail_counts,
)

plots.use_project_style()

frames = load_many()
# Drop the final bar everywhere: if the market is open it is a partial session.
frames = {t: df.iloc[:-1] for t, df in frames.items()}

nflx = frames[config.TICKER]
closes = closing_prices(frames)
rets = closes.pct_change().dropna()

print(f"{len(nflx):,} sessions  |  {nflx.index.min().date()} -> {nflx.index.max().date()}")
print("tickers:", list(closes.columns))

---
## 1. Price and volume over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})

axes[0].plot(nflx.index, nflx["Close"], linewidth=1.1, color="#c0392b")
axes[0].set_title(f"{config.TICKER} adjusted close and volume")
axes[0].set_ylabel("Close (USD, adjusted)")

axes[1].fill_between(nflx.index, nflx["Volume"] / 1e6, color="#7f8c8d", alpha=0.7)
axes[1].set_ylabel("Volume (M)")
axes[1].set_xlabel("Date")

plots.save(fig, "02_price_volume")
plt.show()

Volume spikes line up with the sharpest price moves. That is the market repricing on news, and
it is the visual signature of the events no price-only model can anticipate.

### The two days that define this series

In [ ]:
nflx_ret = daily_returns(nflx["Close"])

worst = nflx_ret.nsmallest(5)
best = nflx_ret.nlargest(5)

print("Five worst days")
for d, v in worst.items():
    print(f"  {d.date()}   {v*100:>7.1f}%   volume {nflx.loc[d, 'Volume']/1e6:>6.1f}M")
print("\nFive best days")
for d, v in best.items():
    print(f"  {d.date()}   {v*100:>7.1f}%   volume {nflx.loc[d, 'Volume']/1e6:>6.1f}M")

**2022-04-20 is the single most important date in this dataset.** Netflix reported its first
subscriber loss in a decade and the stock fell roughly a third in one session.

No model in this project sees subscriber numbers, earnings, or news. Every model here sees only
past prices. So none of them could have predicted that day, and any evaluation that quietly
smooths over it is overstating what the model knows. This is the concrete instance of the warning
ProjectPro gives about news shocks.

---
## 2. Relative performance against peers

Plotting raw prices side by side is misleading — the tickers trade at different levels. Rebasing
every series to 100 at the start makes them comparable.

In [ ]:
rebased = closes / closes.iloc[0] * 100

fig, ax = plt.subplots(figsize=(13, 6))
for ticker in rebased.columns:
    highlight = ticker == config.TICKER
    ax.plot(rebased.index, rebased[ticker],
            linewidth=2.0 if highlight else 1.0,
            alpha=1.0 if highlight else 0.65,
            label=ticker)

ax.set_yscale("log")
ax.set_title("Growth of $100 invested at the start (log scale)")
ax.set_ylabel("Value of $100")
ax.set_xlabel("Date")
ax.legend(ncol=5)

plots.save(fig, "02_relative_performance")
plt.show()

print((rebased.iloc[-1] / 100).round(2).sort_values(ascending=False).to_string())

A log scale is the right choice here: on a log axis, equal vertical distances are equal
*percentage* moves, so a 50% gain looks the same whether it happened at \$10 or \$100. On a
linear axis the recent years would visually dominate and the early years would look flat.

---
## 3. Moving averages

The 10, 20 and 50-day moving averages from the reference notebook. Over ten years they are
indistinguishable from the price line, so the second panel zooms into the last year where they
are actually readable.

In [ ]:
nflx_ma = add_moving_averages(nflx, windows=(10, 20, 50))
ma_cols = ["Close", "MA_10", "MA_20", "MA_50"]

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

nflx_ma[ma_cols].plot(ax=axes[0], linewidth=1.0)
axes[0].set_title(f"{config.TICKER} with 10/20/50-day moving averages — full history")
axes[0].set_ylabel("USD")

last_year = nflx_ma.loc[nflx_ma.index >= nflx_ma.index.max() - pd.Timedelta(days=365)]
last_year[ma_cols].plot(ax=axes[1], linewidth=1.4)
axes[1].set_title("Last 12 months")
axes[1].set_ylabel("USD")
axes[1].set_xlabel("Date")

plt.tight_layout()
plots.save(fig, "02_moving_averages")
plt.show()

Worth noticing: a moving average is **always behind** the price, by construction — it is an
average of days that have already happened. That lag is exactly the failure mode we expect the
LSTM to reproduce in notebook 05, and it is why a chart that "tracks the price well" proves
nothing on its own.

---
## 4. Daily returns

Prices are the wrong thing to model directly. Returns are what is comparable across time and
across tickers, and they are what the stationarity work in notebook 03 depends on.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(nflx_ret.index, nflx_ret * 100, linewidth=0.6, color="#2c3e50")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title(f"{config.TICKER} daily returns (%)")
ax.set_ylabel("Return (%)")
ax.set_xlabel("Date")

plots.save(fig, "02_daily_returns")
plt.show()

This plot shows **volatility clustering** — calm stretches and turbulent stretches arrive in runs
rather than at random. Big moves are followed by big moves.

That matters: it means daily returns are not independent draws. Their *direction* is close to
unpredictable, but their *magnitude* is somewhat predictable. Forecasting volatility is a genuinely
tractable problem; forecasting direction is not. It is worth being clear about which one this
project is attempting.

### Is the return distribution normal?

Nearly every introductory treatment assumes it is. Here is the check.

In [ ]:
clean = nflx_ret.dropna()
mu, sigma = clean.mean(), clean.std()
x = np.linspace(clean.min(), clean.max(), 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(clean, bins=120, density=True, alpha=0.75,
             color="#2980b9", label="observed")
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=1.8,
             label="normal fit")
axes[0].set_title("Daily return distribution")
axes[0].set_xlabel("Daily return")
axes[0].legend()

axes[1].hist(clean, bins=120, density=True, alpha=0.75,
             color="#2980b9", label="observed")
axes[1].plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=1.8,
             label="normal fit")
axes[1].set_yscale("log")
axes[1].set_title("Same data, log scale — the tails become visible")
axes[1].set_xlabel("Daily return")
axes[1].legend()

plt.tight_layout()
plots.save(fig, "02_return_distribution")
plt.show()

On the linear axis the normal fit looks reasonable. On the log axis it clearly is not: the observed
bars extend far past where the red curve has effectively reached zero.

Counting those days makes it concrete.

In [ ]:
print(tail_counts(nflx_ret).to_string(index=False))

**This is the headline result of the EDA.** Extreme days are not rare-but-possible under a normal
distribution — they are orders of magnitude more common than it allows. Any model, risk figure or
confidence interval that assumes normality is understating the chance of a large loss, and it is
understating it by a lot at the far tail.

In [ ]:
summary = pd.DataFrame({t: return_summary(rets[t]) for t in rets.columns}).T
summary_display = summary.copy()
for col in ["mean_daily", "std_daily", "annualized_return", "annualized_volatility",
            "worst_day", "best_day"]:
    summary_display[col] = (summary_display[col] * 100).round(2).astype(str) + "%"
summary_display[["skew", "excess_kurtosis"]] = summary_display[["skew", "excess_kurtosis"]].astype(float).round(2)
summary_display["observations"] = summary_display["observations"].astype(int)
summary_display.T

`excess_kurtosis` is 0 for a normal distribution. Every ticker here is far above it, so fat tails
are a property of equity returns generally, not a quirk of Netflix.

---
## 5. Correlation — and a trap

The reference notebook plots a correlation heatmap of **closing prices**. That number is close to
meaningless, and it is worth showing why rather than just asserting it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(rets.corr(), annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, vmin=-1, vmax=1, ax=axes[0], square=True,
            cbar_kws={"shrink": 0.8})
axes[0].set_title("Correlation of daily RETURNS  (meaningful)")

sns.heatmap(closes.corr(), annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, vmin=-1, vmax=1, ax=axes[1], square=True,
            cbar_kws={"shrink": 0.8})
axes[1].set_title("Correlation of PRICE LEVELS  (misleading)")

plt.tight_layout()
plots.save(fig, "02_correlation")
plt.show()

The right-hand panel reports high correlations between series that need not move together at all.
Both series trend, so both drift upward over the same decade, and correlation picks up the shared
trend rather than any shared behaviour. This is **spurious correlation** between non-stationary
series, and it is the same non-stationarity the ADF test formalises in notebook 03.

The left panel is the honest one, because differencing prices into returns removes the trend.
Read that panel and ignore the right one — it is included only to show the size of the distortion.

### A data-quality caveat on WBD

All five tickers report history back to 2015, but **Warner Bros. Discovery did not exist until
2022** — Yahoo Finance backfills the ticker with its predecessor company. Its pre-2022 series
therefore describes a different business.

Flagging it rather than quietly correlating against it. The cell below quantifies how much the
answer changes if we only use the period since WBD actually existed.

In [ ]:
since_2022 = rets.loc["2022-04-11":]

comparison = pd.DataFrame({
    "full period (2015-)": rets.corr()[config.TICKER],
    "since Apr 2022": since_2022.corr()[config.TICKER],
}).drop(index=config.TICKER).round(3)
comparison["difference"] = (comparison["since Apr 2022"] - comparison["full period (2015-)"]).round(3)
comparison

---
## 6. Risk versus return

Mean daily return on one axis, standard deviation of daily returns on the other. Up is better,
left is safer.

In [ ]:
mean_ret = rets.mean() * 100
risk = rets.std() * 100

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(risk, mean_ret, s=180, alpha=0.75,
           c=["#c0392b" if t == config.TICKER else "#2980b9" for t in rets.columns])

for ticker in rets.columns:
    ax.annotate(ticker, xy=(risk[ticker], mean_ret[ticker]),
                xytext=(12, 12), textcoords="offset points",
                fontsize=11, fontweight="bold",
                arrowprops=dict(arrowstyle="-", color="grey", alpha=0.6))

ax.set_xlabel("Risk — daily return standard deviation (%)")
ax.set_ylabel("Expected daily return (%)")
ax.set_title("Risk versus return, 2015 to present")
ax.axhline(0, color="grey", linewidth=0.8)

plots.save(fig, "02_risk_return")
plt.show()

pd.DataFrame({
    "ann. return %": (rets.mean() * 252 * 100).round(1),
    "ann. volatility %": (rets.std() * np.sqrt(252) * 100).round(1),
    "return per unit risk": (rets.mean() / rets.std() * np.sqrt(252)).round(2),
}).sort_values("return per unit risk", ascending=False)

The final column is a Sharpe-like ratio with the risk-free rate set to zero — return earned per
unit of volatility. It is the fair way to compare: a higher return that came with proportionally
more risk is not obviously better.

SPY is the benchmark. A single stock beating a diversified index on this measure is doing
something genuinely unusual; most do not.

---
## 7. Volatility over time

In [ ]:
vol = rolling_volatility(nflx_ret, window=21) * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(vol.index, vol, linewidth=1.0, color="#8e44ad")
ax.axhline(vol.mean(), color="grey", linestyle="--", linewidth=1.0,
           label=f"mean {vol.mean():.0f}%")
ax.set_title(f"{config.TICKER} 21-day realized volatility, annualized")
ax.set_ylabel("Annualized volatility (%)")
ax.set_xlabel("Date")
ax.legend()

plots.save(fig, "02_volatility")
plt.show()

print(f"mean {vol.mean():.1f}%   min {vol.min():.1f}%   max {vol.max():.1f}%")

Volatility is not constant — it ranges over roughly an order of magnitude. Models that assume a
single fixed error variance across the whole sample, which includes the LSTM as normally trained,
are assuming something the data plainly contradicts.

---
## What this established

1. **Returns are not normally distributed.** Extreme days occur far more often than a normal
   distribution permits — dramatically so in the far tail. Risk estimates built on normality are
   wrong in the direction that matters.
2. **Volatility clusters.** Returns are not independent draws. Magnitude carries some structure
   even though direction largely does not.
3. **Correlating price levels is a trap.** It measures shared trend, not shared behaviour.
   Correlate returns.
4. **The 2022 crash is unpredictable from prices alone**, and every result later in this project
   has to be read with that in mind.
5. **WBD's pre-2022 history is a different company**, so peer comparisons over the full period
   carry a caveat.

**Next:** `03_stationarity_and_features.ipynb` — formalising point 3 with the Augmented
Dickey-Fuller test on prices versus returns, then ACF/PACF to pick candidate ARIMA orders.